# 🛢️ Oil Slick Detection — Pipeline Principal
**TAMI 2026 · YPF · Golfo San Matías, Río Negro**

Este notebook orquesta el pipeline completo de detección automática de manchas
de hidrocarburos en imágenes SAR del satélite Sentinel-1.

| Etapa | Módulo | Descripción |
|-------|--------|-------------|
| 1 | `acquisition.py`    | Descarga imágenes Sentinel-1 filtradas por viento y topografía |
| 2 | `preprocessing.py`  | Filtro Lee + binarización Otsu-Bradley + morfología |
| 3 | `features.py`       | Extracción de 16 características geométricas y texturales |
| 4 | `model.py`          | Inferencia con MLP (PyTorch) |
| 5 | `visualization.py`  | Mapa de calor Folium + auditoría visual (La Lupa) |

---
**Antes de empezar:**
1. Ajustar `GEE_PROJECT` en `src/config.py` con el ID de tu proyecto de Google Cloud.
2. Ver el README para instrucciones completas de configuración de GEE.

## ⚙️ 0. Setup
Monta Drive, crea los directorios necesarios, instala dependencias y autentica GEE.

> **Ejecución local:** comentar el bloque de `google.colab` y ajustar `REPO_PATH`.
> Ver sección *Opción B* del README para configurar rutas y venv.

In [ ]:
# ── Colab: montar Drive ──────────────────────────────────────────────────────
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_PATH = '/content/drive/MyDrive/oil-slick-detection'
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
# ─────────────────────────────────────────────────────────────────────────────

# Instalar dependencias
!pip install -q scikit-image folium rasterio torch joblib earthengine-api

# Crear directorios de trabajo en Colab (SSD local) y en Drive
from pathlib import Path
from src.config import PATHS

for key, path in PATHS.items():
    if not str(path).endswith(('.csv', '.zip', '.pth', '.joblib')):
        Path(path).mkdir(parents=True, exist_ok=True)

print('✅ Directorios creados:')
for key, path in PATHS.items():
    if not str(path).endswith(('.csv', '.zip', '.pth', '.joblib')):
        print(f'   {key}: {path}')

# Autenticar Google Earth Engine
import ee
ee.Authenticate()

from src.config import GEE_PROJECT
ee.Initialize(project=GEE_PROJECT)

print('\n✅ Entorno listo.')

## 🌐 1. Adquisición de imágenes SAR

> ⚠️ **Esta celda se ejecuta UNA SOLA VEZ.** Descarga todos los tiles SAR del año
> configurado en `config.py` y genera el **CSV maestro** (`Registro_Maestro_Golfo.csv`)
> que usan todas las etapas siguientes. Si el CSV ya existe, la descarga retoma
> desde el punto de interrupción (checkpointing automático).
>
> Una vez completada, saltar directamente a la **Etapa 3 (Inferencia)**.

In [ ]:
from src.acquisition import build_water_grid, download_sar_dataset
from src.config import AOI, ACQUISITION, PATHS

# 1a. Generar grilla de puntos marinos (filtra tierra con ETOPO1)
water_coords = build_water_grid(
    lon_min=AOI['lon_min'], lon_max=AOI['lon_max'],
    lat_min=AOI['lat_min'], lat_max=AOI['lat_max'],
    step=AOI['grid_step_deg'],
    elevation_threshold=ACQUISITION['elevation_threshold_m'],
)

# 1b. Descargar imágenes y generar CSV maestro
# Solo descarga tiles con viento ERA5 entre 3-10 m/s.
# Al terminar genera: Registro_Maestro_Golfo.csv
df_maestro = download_sar_dataset(
    water_coords=water_coords,
    local_dir=PATHS['local_tifs'],
    master_csv=PATHS['master_csv'],
)

n_ok = (df_maestro['Estado_Descarga'] == 'OK').sum()
print(f'\n📊 Dataset: {len(df_maestro)} registros totales | {n_ok} imágenes descargadas.')
print(f'📄 CSV maestro guardado en: {PATHS["master_csv"]}')

## 🤖 2. Preprocesamiento + Inferencia con MLP

**Antes de empezar:**
1. Copiar `feature_net_model.pth` y `scaler.joblib` en la carpeta `Modelos/` del Drive.
2. Si es la segunda vez que querés correr el modelo, ejecutar la sección `🧹 Utilidad` previamente para limpiar


Aplica el pipeline completo sobre cada imagen descargada:
1. Normalización SAR (dB → [0, 255])
2. Filtro de Lee (reducción de Speckle)
3. Binarización Otsu-Bradley + morfología
4. Extracción de 16 características
5. Clasificación con MLP → actualiza el CSV maestro con `Prediccion_IA` y `Confianza_IA`

In [ ]:
import shutil
import pandas as pd
from src.model import load_model, run_inference_batch
from src.features import extract_features
from src.config import PATHS, MODEL, PREPROCESSING

# Descomprimir imágenes si vienen en ZIP desde Drive
if not PATHS['local_infer'].exists() or not any(PATHS['local_infer'].rglob('*.tif')):
    if PATHS['images_zip'].exists():
        print(f'📦 Extrayendo {PATHS["images_zip"]}...')
        shutil.unpack_archive(str(PATHS['images_zip']), str(PATHS['local_infer']))
    else:
        PATHS['local_infer'].mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(PATHS['local_tifs']), str(PATHS['local_infer']), dirs_exist_ok=True)

# Cargar modelo y scaler
model, scaler, device = load_model(
    weights_path=PATHS['models_dir'] / MODEL['weights_file'],
    scaler_path =PATHS['models_dir'] / MODEL['scaler_file'],
)
print('✅ Red Neuronal cargada exitosamente en:', device)

# Leer CSV maestro
df_maestro = pd.read_csv(PATHS['master_csv'])

# Inferencia
df_maestro = run_inference_batch(
    df_maestro=df_maestro,
    model=model,
    scaler=scaler,
    device=device,
    tifs_dir=PATHS['local_infer'],
    extract_fn=extract_features,
    max_nodata_fraction=PREPROCESSING['max_nodata_fraction'],
)

df_maestro.to_csv(PATHS['master_csv'], index=False)
print('\n✅ Pipeline finalizado con éxito. Red Neuronal aplicada y CSV actualizado.')

## 🗺️ 3. Mapa de Calor

Al hacer clic en una de las coordenadas, se muestra el resumen de la misma

Se puede aprovechar la celda 4 para auditar coordenadas manualmente.

In [ ]:
# --- 3. Generar y mostrar el mapa ---
from src.visualization import build_heatmap
from src.config import PATHS, VIZ
import pandas as pd

df_maestro = pd.read_csv(PATHS['master_csv'])

mapa = build_heatmap(
    master_csv=df_maestro,
    output_path=PATHS['images_dir'].parent / VIZ['output_map_filename']
    )

mapa

## 🔍 4. La Lupa manual

Alternativa al botón del mapa. Ingresá las coordenadas de un punto del mapa de calor
para generar el expediente visual completo:
- `_1_DetalleOriginal.jpg` — imagen SAR normalizada
- `_2_MascaraMLP.jpg` — máscara binaria Otsu-Bradley
- `_3_ContextoPanoramico_15km.jpg` — contexto de 15×15 km desde GEE

Útil en **uso local** o para auditar una coordenada específica.

In [ ]:
## 🔍 4. La Lupa interactiva
import ipywidgets as widgets
from IPython.display import display
from src.visualization import extract_evidence
from src.preprocessing import lee_filter, apply_mixed_filter
from src.config import PATHS, GEE_PROJECT

opciones = [
    (f"{lat:.3f}, {lon:.3f} — {len(g)} det. — {g['Confianza_IA'].mean():.0f}%", (lon, lat))
    for (lat, lon), g in df_maestro[df_maestro["Prediccion_IA"] == "Slick Petroleo"].groupby(["Lat", "Lon"])
]

dd  = widgets.Dropdown(options=sorted(opciones), description="📍", layout=widgets.Layout(width="420px"))
btn = widgets.Button(description="🔍 La Lupa", button_style="primary")
out = widgets.Output()

def _run(_):
    out.clear_output()
    lon, lat = dd.value
    with out:
        extract_evidence(lon, lat, df_maestro, PATHS["local_infer"],
                         PATHS["evidence_dir"] / f"Exp_Lon{lon:.3f}_Lat{lat:.3f}",
                         lee_filter, apply_mixed_filter, GEE_PROJECT)

btn.on_click(_run)
display(widgets.VBox([widgets.HBox([dd, btn]), out]))

---
## 🧹 Utilidad: resetear predicciones para re-inferencia

In [ ]:
import pandas as pd
from src.config import PATHS

df = pd.read_csv(PATHS['master_csv'])
n_old = (df['Prediccion_IA'] != 'Pendiente').sum()
df.loc[df['Estado_Descarga'] == 'OK', 'Prediccion_IA'] = 'Pendiente'
df.loc[df['Estado_Descarga'] == 'OK', 'Confianza_IA']  = 0.0
df.to_csv(PATHS['master_csv'], index=False)
print(f'✅ Se resetearon {n_old} predicciones. Listo para re-inferir.')